In [3]:
!pip install -U transformers accelerate bitsandbytes


In [4]:
import torch
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. Manually set your token
my_hf_token = ""

# 2. Log in
login(my_hf_token)

# 3. Load Gemma 2 2B
model_id = "google/gemma-2-2b-it"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="cpu", # Changed from "auto" to "cpu"
    torch_dtype=torch.float32
)

print("Model loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Model loaded successfully!


In [5]:
for name, param in list(model.named_parameters())[:15]:
    print(f"{name:60s}  {str(param.shape):25s}")

model.embed_tokens.weight                                     torch.Size([256000, 2304])
model.layers.0.self_attn.q_proj.weight                        torch.Size([2048, 2304]) 
model.layers.0.self_attn.k_proj.weight                        torch.Size([1024, 2304]) 
model.layers.0.self_attn.v_proj.weight                        torch.Size([1024, 2304]) 
model.layers.0.self_attn.o_proj.weight                        torch.Size([2304, 2048]) 
model.layers.0.mlp.gate_proj.weight                           torch.Size([9216, 2304]) 
model.layers.0.mlp.up_proj.weight                             torch.Size([9216, 2304]) 
model.layers.0.mlp.down_proj.weight                           torch.Size([2304, 9216]) 
model.layers.0.input_layernorm.weight                         torch.Size([2304])       
model.layers.0.post_attention_layernorm.weight                torch.Size([2304])       
model.layers.0.pre_feedforward_layernorm.weight               torch.Size([2304])       
model.layers.0.post_feedforward

In [6]:
mlp_gate = model.model.layers[0].mlp.gate_proj.weight
row_norms = mlp_gate.norm(dim=1)

In [7]:
print(f"Mean row norm:  {row_norms.mean():.4f}")
print(f"Std row norm:   {row_norms.std():.4f}")
print(f"Max row norm:   {row_norms.max():.4f}")
print(f"Min row norm:   {row_norms.min():.4f}")

Mean row norm:  0.3928
Std row norm:   0.0443
Max row norm:   0.5676
Min row norm:   0.1838


In [8]:
def get_hidden_states(text):
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    return outputs.hidden_states

In [9]:
text   = "def binary_search(arr, target):"
states = get_hidden_states(text)
last_token_norms = [
    states[i][0, -1, :].norm().item()
    for i in range(len(states))
]
for i, norm in enumerate(last_token_norms):
    bar = "█" * int(norm / max(last_token_norms) * 30)
    print(f"Layer {i:2d}: {norm:.3f}  {bar}")

Layer  0: 76.465  ███
Layer  1: 82.893  ████
Layer  2: 86.383  ████
Layer  3: 79.910  ███
Layer  4: 81.269  ████
Layer  5: 82.372  ████
Layer  6: 105.731  █████
Layer  7: 98.442  ████
Layer  8: 117.302  █████
Layer  9: 127.627  ██████
Layer 10: 143.604  ███████
Layer 11: 158.782  ███████
Layer 12: 205.706  ██████████
Layer 13: 197.916  █████████
Layer 14: 222.772  ██████████
Layer 15: 238.521  ███████████
Layer 16: 273.248  █████████████
Layer 17: 281.673  █████████████
Layer 18: 251.842  ████████████
Layer 19: 291.011  ██████████████
Layer 20: 361.293  █████████████████
Layer 21: 380.903  ██████████████████
Layer 22: 411.840  ████████████████████
Layer 23: 470.569  ███████████████████████
Layer 24: 522.569  █████████████████████████
Layer 25: 608.890  ██████████████████████████████
Layer 26: 110.962  █████


In [10]:
def head_effective_rank(layer_idx, head_idx, threshold=0.90):
    layer    = model.model.layers[layer_idx]
    W_q      = layer.self_attn.q_proj.weight        # [2048, 2304]
    head_dim = 256
    W_q_head = W_q[head_idx * head_dim:(head_idx + 1) * head_dim, :].to('cpu') # Move to CPU
    _, S, _ = torch.linalg.svd(W_q_head, full_matrices=False)
    cumvar  = torch.cumsum(S**2, 0) / (S**2).sum()
    rank    = (cumvar < threshold).sum().item() + 1
    return rank

In [11]:
print("Layer | Head | Eff. Rank | Interpretation")
print("-" * 55)
for layer in [0, 6, 13, 20, 25]:
    for head in [0, 4, 7]:
        r    = head_effective_rank(layer, head)
        kind = "local/syntactic" if r < 30 else (
               "transition" if r < 80 else "semantic/global")
        print(f"  {layer:2d}  |   {head}  |    {r:3d}    | {kind}")

Layer | Head | Eff. Rank | Interpretation
-------------------------------------------------------
   0  |   0  |    177    | semantic/global
   0  |   4  |    188    | semantic/global
   0  |   7  |    185    | semantic/global
   6  |   0  |    191    | semantic/global
   6  |   4  |    105    | semantic/global
   6  |   7  |    172    | semantic/global
  13  |   0  |    142    | semantic/global
  13  |   4  |    145    | semantic/global
  13  |   7  |    165    | semantic/global
  20  |   0  |    192    | semantic/global
  20  |   4  |    128    | semantic/global
  20  |   7  |    115    | semantic/global
  25  |   0  |    179    | semantic/global
  25  |   4  |    137    | semantic/global
  25  |   7  |    180    | semantic/global


In [12]:
def mlp_memory_audit(layer_idx, top_k=10):
    layer  = model.model.layers[layer_idx]
    W_gate = layer.mlp.gate_proj.weight   # [9216, 2304] - keys
    W_down = layer.mlp.down_proj.weight

    # How impactful is each memory slot?
    read_strength  = W_gate.norm(dim=1)   # how strongly it responds to input
    write_strength = W_down.norm(dim=0)   # how strongly it writes to residual stream
    importance     = read_strength * write_strength
    top_slots = torch.topk(importance, k=top_k).indices
    print(f"\nLayer {layer_idx} - top {top_k} memory slots:")
    for slot in top_slots[:5]:
        key = W_gate[slot]
        val = W_down[:, slot]
        cos = torch.nn.functional.cosine_similarity(
            key.unsqueeze(0), val.unsqueeze(0)
        ).item()
        print(f"  Slot {slot.item():5d} | "
              f"read={read_strength[slot]:.3f} | "
              f"write={write_strength[slot]:.3f} | "
              f"key·val cosine={cos:+.3f}")

In [13]:
for l in [0, 6, 13, 20, 25]:
    mlp_memory_audit(l)


Layer 0 - top 10 memory slots:
  Slot  4056 | read=0.527 | write=0.517 | key·val cosine=+0.185
  Slot  2183 | read=0.447 | write=0.596 | key·val cosine=-0.134
  Slot  1858 | read=0.445 | write=0.566 | key·val cosine=+0.066
  Slot  1827 | read=0.501 | write=0.501 | key·val cosine=+0.378
  Slot    79 | read=0.568 | write=0.420 | key·val cosine=+0.291

Layer 6 - top 10 memory slots:
  Slot  8196 | read=0.757 | write=0.384 | key·val cosine=-0.469
  Slot  1123 | read=0.646 | write=0.413 | key·val cosine=+0.065
  Slot  1940 | read=0.531 | write=0.475 | key·val cosine=+0.014
  Slot  6198 | read=0.560 | write=0.434 | key·val cosine=-0.181
  Slot  6747 | read=0.515 | write=0.462 | key·val cosine=+0.457

Layer 13 - top 10 memory slots:
  Slot  1643 | read=0.824 | write=0.358 | key·val cosine=+0.003
  Slot  4433 | read=0.905 | write=0.326 | key·val cosine=+0.610
  Slot  5773 | read=0.942 | write=0.294 | key·val cosine=-0.423
  Slot   109 | read=0.715 | write=0.381 | key·val cosine=-0.193
  Slot 

In [14]:
def neuron_identity_check(layer_idx, neuron_idx, texts):
    scores = []
    for text in texts:
        inputs = tokenizer(text, return_tensors="pt")
        with torch.no_grad():
            out = model(**inputs, output_hidden_states=True)
        hidden = out.hidden_states[layer_idx]
        gate   = model.model.layers[layer_idx].mlp.gate_proj(hidden)
        act    = torch.nn.functional.silu(gate)
        score  = act[0, :, neuron_idx].mean().item()
        scores.append((score, text[:80]))

    scores.sort(reverse=True)
    print(f"\nNeuron {neuron_idx} @ Layer {layer_idx} - top activating inputs:")
    for score, text in scores[:8]:
        print(f"  {score:.4f}  |  {text}")

In [15]:
diverse_texts = [
    "def quicksort(arr): pivot = arr[len(arr)//2]",
    "Le soleil couchant illumine les toits de Paris",     # French poetry
    "    if indent_level > 0:\n        pass",             # Python indentation
    "The mitochondria is the powerhouse of the cell",
    "for i in range(n): total += arr[i]",
    "Baudelaire écrit dans Les Fleurs du Mal",            # French literature
    "class BinaryTree:\n    def __init__(self):",
    "The Jacobian matrix encodes partial derivatives",
    "    return {\n        'key': value\n    }",          # nested indentation
    "Victor Hugo a publié Les Misérables en 1862",        # French history
]
neuron_identity_check(layer_idx=13, neuron_idx=4721, texts=diverse_texts)


Neuron 4721 @ Layer 13 - top activating inputs:
  0.4643  |  Victor Hugo a publié Les Misérables en 1862
  0.3298  |  The Jacobian matrix encodes partial derivatives
  0.2634  |  Baudelaire écrit dans Les Fleurs du Mal
  0.2556  |  def quicksort(arr): pivot = arr[len(arr)//2]
  0.1131  |  The mitochondria is the powerhouse of the cell
  0.0801  |  class BinaryTree:
    def __init__(self):
  0.0687  |  for i in range(n): total += arr[i]
  -0.0023  |      return {
        'key': value
    }


In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np

In [17]:
def probe_by_layer(texts, labels):
    all_activations = [[] for _ in range(27)]
    for text in texts:
        states = get_hidden_states(text)
        for i, state in enumerate(states):
            all_activations[i].append(state[0, -1, :].float().numpy())
    print("\nLayer  |  Accuracy  |  Signal")
    print("-" * 40)
    for layer_idx in range(0, 27, 2):
        X = np.array(all_activations[layer_idx])
        y = np.array(labels)
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)
        clf = LogisticRegression(max_iter=500).fit(X_tr, y_tr)
        acc = accuracy_score(y_te, clf.predict(X_te))
        bar = "▓" * int(acc * 20)
        print(f"   {layer_idx:2d}  |   {acc:.2f}    |  {bar}")
# Probe: does this snippet contain a loop?
snippets = [
    "def f(x): return x * 2",
    "result = sum(values)",
    "for i in range(n): total += i",
    "while queue: process(queue.pop())",
    "x = [v**2 for v in data]",
    "return max(a, b)",
    "node = node.next",
    "for item in batch: yield item",
]
loop_labels = [0, 0, 1, 1, 1, 0, 0, 1]
probe_by_layer(snippets, loop_labels)


Layer  |  Accuracy  |  Signal
----------------------------------------
    0  |   0.33    |  ▓▓▓▓▓▓
    2  |   0.67    |  ▓▓▓▓▓▓▓▓▓▓▓▓▓
    4  |   0.67    |  ▓▓▓▓▓▓▓▓▓▓▓▓▓
    6  |   0.33    |  ▓▓▓▓▓▓
    8  |   0.33    |  ▓▓▓▓▓▓
   10  |   0.33    |  ▓▓▓▓▓▓
   12  |   0.33    |  ▓▓▓▓▓▓
   14  |   0.33    |  ▓▓▓▓▓▓
   16  |   0.33    |  ▓▓▓▓▓▓
   18  |   0.67    |  ▓▓▓▓▓▓▓▓▓▓▓▓▓
   20  |   0.67    |  ▓▓▓▓▓▓▓▓▓▓▓▓▓
   22  |   0.67    |  ▓▓▓▓▓▓▓▓▓▓▓▓▓
   24  |   0.67    |  ▓▓▓▓▓▓▓▓▓▓▓▓▓
   26  |   0.33    |  ▓▓▓▓▓▓


In [18]:
def layer_similarity_audit():
    gate_vectors = []
    for layer in model.model.layers:
        W = layer.mlp.gate_proj.weight.float().reshape(-1)
        gate_vectors.append(W / W.norm())

    n = len(gate_vectors)
    print("Pairwise cosine similarity (selected layer pairs):")
    print(f"{'Pair':15s}  {'Similarity':>12s}  {'Verdict'}")
    print("-" * 55)
    pairs = [(i, i+1) for i in range(0, 25, 3)] + [(0,25), (5,20), (10,15)]
    for (i, j) in pairs:
        sim = torch.dot(gate_vectors[i], gate_vectors[j]).item()
        verdict = ("near-identical - redundant?" if sim > 0.7 else
                   "similar" if sim > 0.4 else
                   "distinct - both earning their place")
        print(f"Layer {i:2d} vs {j:2d}   {sim:12.4f}  {verdict}")

In [ ]:
layer_similarity_audit()